In [2]:
import json
import random
from datasets import load_dataset

# 1. 下载并加载数据集 (开启 streaming=True)
print("正在以流式模式连接数据集...")
# 加载 ReasonLite-Dataset (数学)
ds_math_stream = load_dataset("amd/ReasonLite-Dataset", split="medium", streaming=True)
# 加载 Alpaca (通用指令)
ds_alp_stream = load_dataset("tatsu-lab/alpaca", split="train", streaming=True)

# 2. 设置参数
MATH_SAMPLES = 50000 
MATH_EVAL_SAMPLES = 1000  # 对应你原代码中的 1.1倍逻辑，取 1000 条做验证集
GENERAL_SAMPLES = 2000
TOTAL_MATH_NEEDED = MATH_SAMPLES + MATH_EVAL_SAMPLES

# 固定随机种子
SEED = 40

# 3. 提取并转换格式
# 在流式模式下，我们需要先打乱(shuffle)再提取(take)
# buffer_size 是缓冲区大小，越大随机性越好，但占用内存更多
print(f"正在从流中提取 {TOTAL_MATH_NEEDED} 条数学数据...")
shuffled_math = ds_math_stream.shuffle(seed=SEED, buffer_size=10000)
math_list = []

# 迭代提取数学数据
for i, item in enumerate(shuffled_math.take(TOTAL_MATH_NEEDED)):
    # 注意：ReasonLite 的字段名通常是 'problem' 和 'solution'
    # 如果字段名不对，请根据实际情况修改
    new_item = {
        "query": item['prompt'],   # 获取原始数据的 prompt 列
        "response": item['answer'], # 获取原始数据的 answer 列
        "type": "ReasonLite"
    }
    math_list.append(new_item)

print(f"正在从流中提取 {GENERAL_SAMPLES} 条 Alpaca 数据...")
shuffled_alp = ds_alp_stream.shuffle(seed=SEED, buffer_size=5000)
alp_list = []

for item in shuffled_alp.take(GENERAL_SAMPLES):
    new_item = {
        "query": item['instruction'],
        "response": item['output'],
        "type": "Alpaca"
    }
    alp_list.append(new_item)

# 4. 选取子集并切分 (模拟你之前的逻辑)
print("正在处理训练集和验证集...")

math_train_subset = math_list[:MATH_SAMPLES]
train_data = math_train_subset + alp_list
random.seed(SEED)
random.shuffle(train_data) # 混合打乱

# 验证集：数学数据的后续部分 (从 10000 到 11000)
valid_data = math_list[MATH_SAMPLES:TOTAL_MATH_NEEDED]

print(f"处理完成：训练集 {len(train_data)} 条，验证集 {len(valid_data)} 条")

# 5. 保存为 JSON 文件
def save_to_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"已成功保存至: {filename}")

print("正在保存文件...")
save_to_json(train_data, 'reasonlite_train_50k.json')
save_to_json(valid_data, 'reasonlite_valid_50k.json')

print("所有操作已完成！(仅下载了必要的少量数据)")

正在以流式模式连接数据集...


Resolving data files:   0%|          | 0/92 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/217 [00:00<?, ?it/s]

正在从流中提取 51000 条数学数据...
正在从流中提取 2000 条 Alpaca 数据...
正在处理训练集和验证集...
处理完成：训练集 52000 条，验证集 1000 条
正在保存文件...
已成功保存至: reasonlite_train_50k.json
已成功保存至: reasonlite_valid_50k.json
所有操作已完成！(仅下载了必要的少量数据)


In [4]:
import json
import random
from datasets import load_dataset

# 1. 下载并加载数据集 (开启 streaming=True)
print("正在以流式模式连接数据集...")
# 加载 ReasonLite-Dataset (数学)
ds_math_stream = load_dataset("amd/ReasonLite-Dataset", split="medium", streaming=True)

# 2. 设置参数
MATH_SAMPLES = 10000 
TOTAL_MATH_NEEDED = MATH_SAMPLES

# 固定随机种子
SEED = 40

# 3. 提取并转换格式
# 在流式模式下，我们需要先打乱(shuffle)再提取(take)
# buffer_size 是缓冲区大小，越大随机性越好，但占用内存更多
print(f"正在从流中提取 {TOTAL_MATH_NEEDED} 条数学数据...")
shuffled_math = ds_math_stream.shuffle(seed=SEED, buffer_size=10000)
math_list = []

# 迭代提取数学数据
for i, item in enumerate(shuffled_math.take(TOTAL_MATH_NEEDED)):
    # 注意：ReasonLite 的字段名通常是 'problem' 和 'solution'
    # 如果字段名不对，请根据实际情况修改
    new_item = {
        "query": item['prompt'],   # 获取原始数据的 prompt 列
        # "response": item['answer'], # 获取原始数据的 answer 列
        'expected': item['expected_answer'],
        'vote': item['vote'],
        'source': item['problem_source'],
        "type": "ReasonLite"
    }
    math_list.append(new_item)


# 4. 选取子集并切分 (模拟你之前的逻辑)
print("正在处理训练集和验证集...")

train_data = math_list
random.seed(SEED)
random.shuffle(train_data) # 混合打乱


# 5. 保存为 JSON 文件
def save_to_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"已成功保存至: {filename}")

print("正在保存文件...")
save_to_json(train_data, 'week6-data/reasonlite_10k.json')

print("所有操作已完成！(仅下载了必要的少量数据)")

正在以流式模式连接数据集...


Resolving data files:   0%|          | 0/92 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/217 [00:00<?, ?it/s]

正在从流中提取 10000 条数学数据...
正在处理训练集和验证集...
正在保存文件...
已成功保存至: week6-data/reasonlite_10k.json
所有操作已完成！(仅下载了必要的少量数据)
